In [1]:
import os
print(os.getcwd())

# architect-python-batch

/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/ejercicio_cuatro


In [1]:
%%writefile app.py
import time
import os

def run_batch():
    # El eje inferior izquierdo: Variables de entorno inyectadas por Kubernetes
    batch_name = os.getenv("BATCH_NAME", "Lote-Default")
    total_items = int(os.getenv("TOTAL_ITEMS", "5"))
    
    print(f"=== INICIANDO: {batch_name} ===")
    print("Estado: Punto de equilibrio alcanzado. Procesando...")
    
    for i in range(1, total_items + 1):
        print(f" -> Procesando elemento {i}/{total_items}...")
        time.sleep(1)
        
    print(f"=== TRABAJO COMPLETADO CON ÉXITO ===")

if __name__ == "__main__":
    run_batch()

Writing app.py


In [2]:
%%writefile pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: python-batch-pod
  labels:
    component: architect-python-batch
    layer: exercise-1
spec:
  restartPolicy: Never
  containers:
  - name: python-worker
    image: python:3.10-slim
    command: ["python", "-c", "import os; print('Simulación de entorno lista')"]
    # En el siguiente paso acoplaremos el script completo, 
    # por ahora probemos que la estructura base levanta correctamente.

Writing pod.yaml


In [3]:
%%bash
# Verificar conexión
kubectl cluster-info

# Aplicar el manifiesto base
kubectl apply -f pod.yaml

Kubernetes control plane is running at https://kubernetes.docker.internal:6443
CoreDNS is running at https://kubernetes.docker.internal:6443/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.
pod/python-batch-pod created


In [4]:
%%bash
kubectl get pods -w

NAME                         READY   STATUS      RESTARTS   AGE
batch-job-inicial-jw2pf      0/1     Completed   0          151m
batch-telemetria-job-kpjbv   0/1     Completed   0          32m
python-batch-pod             0/1     Completed   0          26s
Process is interrupted.


In [ ]:
python-batch-pod             0/1     Completed   0          26s

In [5]:
%%bash
kubectl logs python-batch-pod

Simulación de entorno lista


In [6]:
%%bash
kubectl delete -f pod.yaml

pod "python-batch-pod" deleted


In [7]:
%%bash
kubectl create configmap python-script-config --from-file=app.py

configmap/python-script-config created


In [8]:
%%bash
kubectl describe configmap python-script-config

Name:         python-script-config
Namespace:    default
Labels:       <none>
Annotations:  <none>

Data
====
app.py:
----
import time
import os

def run_batch():
    # El eje inferior izquierdo: Variables de entorno inyectadas por Kubernetes
    batch_name = os.getenv("BATCH_NAME", "Lote-Default")
    total_items = int(os.getenv("TOTAL_ITEMS", "5"))
    
    print(f"=== INICIANDO: {batch_name} ===")
    print("Estado: Punto de equilibrio alcanzado. Procesando...")
    
    for i in range(1, total_items + 1):
        print(f" -> Procesando elemento {i}/{total_items}...")
        time.sleep(1)
        
    print(f"=== TRABAJO COMPLETADO CON ÉXITO ===")

if __name__ == "__main__":
    run_batch()


BinaryData
====

Events:  <none>


In [9]:
%%writefile pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: python-batch-pod
  labels:
    component: architect-python-batch
    layer: exercise-1
spec:
  restartPolicy: Never
  containers:
  - name: python-worker
    image: python:3.10-slim
    # Ejecutamos el archivo que montamos en la carpeta /app
    command: ["python", "/app/app.py"]
    
    # El eje inferior izquierdo de la X: Inyección de variables de entorno
    env:
    - name: BATCH_NAME
      value: "Lote-Kubernetes-ConfigMap"
    - name: TOTAL_ITEMS
      value: "7"
      
    # Montamos el volumen en el sistema de archivos del contenedor
    volumeMounts:
    - name: script-volume
      mountPath: /app
  
  # Declaramos el volumen apuntando al ConfigMap
  volumes:
  - name: script-volume
    configMap:
      name: python-script-config

Overwriting pod.yaml


In [10]:
%%bash
kubectl apply -f pod.yaml

pod/python-batch-pod created


In [11]:
%%bash
kubectl get pods python-batch-pod

NAME               READY   STATUS      RESTARTS   AGE
python-batch-pod   0/1     Completed   0          21s


In [12]:
%%bash
kubectl logs python-batch-pod

=== INICIANDO: Lote-Kubernetes-ConfigMap ===
Estado: Punto de equilibrio alcanzado. Procesando...
 -> Procesando elemento 1/7...
 -> Procesando elemento 2/7...
 -> Procesando elemento 3/7...
 -> Procesando elemento 4/7...
 -> Procesando elemento 5/7...
 -> Procesando elemento 6/7...
 -> Procesando elemento 7/7...
=== TRABAJO COMPLETADO CON ÉXITO ===


In [13]:
%%writefile job.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: python-batch-job
  labels:
    component: architect-python-batch
    layer: exercise-1
spec:
  # Intentos de reejecución en caso de que el código falle por un bug interno (Error 1, etc.)
  backoffLimit: 2
  
  template:
    metadata:
      name: python-batch-job-pod
    spec:
      restartPolicy: Never
      containers:
      - name: python-worker
        image: python:3.10-slim
        command: ["python", "/app/app.py"]
        env:
        - name: BATCH_NAME
          value: "Lote-Nativo-Job-Kubernetes"
        - name: TOTAL_ITEMS
          value: "5"
        volumeMounts:
        - name: script-volume
          mountPath: /app
      volumes:
      - name: script-volume
        configMap:
          name: python-script-config

Writing job.yaml


In [14]:
%%bash
kubectl apply -f job.yaml

job.batch/python-batch-job created


In [15]:
%%bash
kubectl get jobs,pods -l component=architect-python-batch

NAME                         COMPLETIONS   DURATION   AGE
job.batch/python-batch-job   1/1           13s        14s

NAME                   READY   STATUS      RESTARTS   AGE
pod/python-batch-pod   0/1     Completed   0          3m46s


In [16]:
%%bash
kubectl logs job/python-batch-job

=== INICIANDO: Lote-Nativo-Job-Kubernetes ===
Estado: Punto de equilibrio alcanzado. Procesando...
 -> Procesando elemento 1/5...
 -> Procesando elemento 2/5...
 -> Procesando elemento 3/5...
 -> Procesando elemento 4/5...
 -> Procesando elemento 5/5...
=== TRABAJO COMPLETADO CON ÉXITO ===


In [17]:
%%bash
kubectl delete -f job.yaml

job.batch "python-batch-job" deleted


In [18]:
%%writefile app.py
import time
import os
import sys

def run_batch():
    batch_name = os.getenv("BATCH_NAME", "Lote-Resiliente")
    total_items = int(os.getenv("TOTAL_ITEMS", "5"))
    simular_error = os.getenv("SIMULAR_ERROR", "False") == "True"
    
    print(f"=== INICIANDO: {batch_name} ===")
    
    for i in range(1, total_items + 1):
        # Provocamos una falla artificial en la iteración 3 si la bandera está activa
        if simular_error and i == 3:
            print("⚠️ [ERROR SIMULADO] El contenedor experimentó una falla crítica en el nodo.")
            sys.exit(1) # Código de salida de error, rompe el contenedor
            
        print(f" -> Procesando elemento {i}/{total_items}...")
        time.sleep(1)
        
    print(f"=== TRABAJO COMPLETADO CON ÉXITO ===")

if __name__ == "__main__":
    run_batch()

Overwriting app.py


In [19]:
%%bash
kubectl delete configmap python-script-config
kubectl create configmap python-script-config --from-file=app.py

configmap "python-script-config" deleted
configmap/python-script-config created


In [20]:
%%writefile job.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: python-batch-job
  labels:
    component: architect-python-batch
    layer: exercise-1
spec:
  # Límite de reintentos antes de marcar el Job como fallido por completo
  backoffLimit: 2
  
  template:
    metadata:
      # IMPORTANTE: Aquí se heredan las etiquetas para el monitoreo
      labels:
        component: architect-python-batch
        layer: exercise-1
    spec:
      restartPolicy: Never
      containers:
      - name: python-worker
        image: python:3.10-slim
        command: ["python", "/app/app.py"]
        env:
        - name: BATCH_NAME
          value: "Lote-Prueba-Falla"
        - name: TOTAL_ITEMS
          value: "5"
        - name: SIMULAR_ERROR
          value: "True"
        volumeMounts:
        - name: script-volume
          mountPath: /app
      volumes:
      - name: script-volume
        configMap:
          name: python-script-config

Overwriting job.yaml


In [21]:
%%bash
kubectl apply -f job.yaml

job.batch/python-batch-job created


In [23]:
%%bash
kubectl get jobs,pods -l component=architect-python-batch

NAME                         COMPLETIONS   DURATION   AGE
job.batch/python-batch-job   0/1           2m38s      2m38s

NAME                         READY   STATUS      RESTARTS   AGE
pod/python-batch-job-6796v   0/1     Error       0          2m38s
pod/python-batch-job-szqdb   0/1     Error       0          2m29s
pod/python-batch-job-xr7r5   0/1     Error       0          2m20s
pod/python-batch-pod         0/1     Completed   0          10m


In [24]:
%%bash
kubectl logs pod/python-batch-job-6796v

=== INICIANDO: Lote-Prueba-Falla ===
 -> Procesando elemento 1/5...
 -> Procesando elemento 2/5...
⚠️ [ERROR SIMULADO] El contenedor experimentó una falla crítica en el nodo.


In [25]:
%%bash
kubectl delete -f job.yaml
kubectl delete configmap python-script-config

job.batch "python-batch-job" deleted
configmap "python-script-config" deleted
